# Remarks and Conclusion

In [ ]:
import pandas as pd
from pathlib import Path
from IPython.display import Image, display

PROJECT_ROOT = Path.cwd().parent
MODEL_OUTPUT_DIR = PROJECT_ROOT / "outputs" / "modeling"
EDA_OUTPUT_DIR = PROJECT_ROOT / "outputs" / "eda"
NN_OUTPUT_DIR = MODEL_OUTPUT_DIR / "neural_network"
LOGREG_OUTPUT_DIR = MODEL_OUTPUT_DIR / "logistic_regression"

## Results

### EDA Results

The EDA suggested that the platforms differ in structure as well as vocabulary. Twitter posts were shorter and much heavier in hashtags, mentions, and URLs. Reddit and Hacker News both had longer discussion tails, but Reddit looked more question-heavy while Hacker News appeared more restrained in punctuation and more analytical in tone. These patterns justified keeping text length and engineered text statistics as real modeling features rather than treating them as noise.

In [ ]:
eda_graphs = [
    ("Class Balance", EDA_OUTPUT_DIR / "class_balance.png"),
    ("Text Length Distributions", EDA_OUTPUT_DIR / "text_length_hist.png"),
    ("Average Word Length by Platform", EDA_OUTPUT_DIR / "style_avg_word_len_bar.png"),
    ("Stylistic Marker Counts by Platform", EDA_OUTPUT_DIR / "style_stats_bar.png"),
    ("Posts by Year-Month and Platform", EDA_OUTPUT_DIR / "year_month_counts.png"),
    ("PCA of Engineered Text Statistics", EDA_OUTPUT_DIR / "pca_stats_scatter.png"),
]

for title, image_path in eda_graphs:
    print(f"\n## {title}")
    if image_path.exists():
        display(Image(filename=str(image_path)))
    else:
        print(f"Missing graph: {image_path}")

style_summary_path = EDA_OUTPUT_DIR / "style_summary.csv"
if style_summary_path.exists():
    print("\nEDA style summary:")
    display(pd.read_csv(style_summary_path))

class_balance_path = EDA_OUTPUT_DIR / "class_balance_summary.csv"
if class_balance_path.exists():
    print("\nEDA class balance summary:")
    display(pd.read_csv(class_balance_path))

### Main Model Results

The logistic-regression baseline reached about **0.879** test accuracy and **0.880** macro F1, showing that surface-level word usage and engineered text statistics already separate the platforms reasonably well. XGBoost on MiniLM sentence embeddings improved on that baseline to about **0.892** test accuracy, suggesting that dense semantic representations and nonlinear decision boundaries capture additional platform-style differences. The neural network performed best overall, reaching about **0.943** test accuracy.

The confusion matrices show the same pattern. Logistic regression has the most confusion between Reddit and Hacker News, XGBoost reduces some of that overlap, and the neural network separates the three classes most cleanly. The neural-network loss and accuracy curves also show stable convergence under early stopping, which supports the final architecture choice.

In [ ]:
model_graphs = [
    ("Logistic Regression Confusion Matrix", MODEL_OUTPUT_DIR / "logreg_confusion_matrix.png"),
    ("XGBoost Confusion Matrix", MODEL_OUTPUT_DIR / "xgboost_embedding_confusion_matrix.png"),
    ("Neural Network Confusion Matrix", NN_OUTPUT_DIR / "nn_confusion_matrix.png"),
    ("Neural Network Loss Curve", NN_OUTPUT_DIR / "nn_loss_curve.png"),
    ("Neural Network Accuracy Curve", NN_OUTPUT_DIR / "nn_accuracy_curve.png"),
]

for title, image_path in model_graphs:
    print(f"\n## {title}")
    if image_path.exists():
        display(Image(filename=str(image_path)))
    else:
        print(f"Missing graph: {image_path}")

### Feature Importance Results

The feature-importance results line up with the EDA. In the logistic model, Twitter is strongly associated with links, mentions, and short high-signal surface markers such as `http`, `https`, and `num_mentions`. Reddit is associated more with longer conversational or advice-seeking text, including features like `num_words` and words such as `help` and `looking`. Hacker News is associated more with article-style or discussion-oriented language. The XGBoost dimension examples support the same story at a higher level: some important embedding directions separate short tagged or promotional posts from longer explanatory discussion, while others separate casual help-seeking text from more analytical commentary.

In [ ]:
logreg_top_features_path = LOGREG_OUTPUT_DIR / "logreg_top_features.csv"
if logreg_top_features_path.exists():
    print("Top logistic-regression features by class:")
    display(pd.read_csv(logreg_top_features_path))
else:
    print(f"Missing logistic feature summary: {logreg_top_features_path}")

xgb_top_dims_path = MODEL_OUTPUT_DIR / "xgboost_top_dimensions.csv"
if xgb_top_dims_path.exists():
    print("\nTop XGBoost embedding dimensions:")
    display(pd.read_csv(xgb_top_dims_path))
else:
    print(f"Missing XGBoost dimension summary: {xgb_top_dims_path}")

xgb_examples_path = MODEL_OUTPUT_DIR / "xgboost_top_dimension_examples.csv"
if xgb_examples_path.exists():
    print("\nExample posts for top XGBoost dimensions:")
    display(pd.read_csv(xgb_examples_path))
else:
    print(f"Missing XGBoost example summary: {xgb_examples_path}")

## Concepts Applied

This project most clearly applies these rubric concepts: **feature engineering**, **feature importance / interpretation**, and **hyperparameter tuning**. Feature engineering appears in the text statistics such as mentions, hashtags, URLs, question counts, word counts, and average word length. Feature importance appears in two forms: readable logistic-regression coefficients and example-based interpretation of important XGBoost embedding dimensions. Hyperparameter tuning appears in all three models through small validation-based sweeps before the final training runs.

The project also applies at least six course topics in relevant ways: **Polars**, **SQL/DuckDB**, **text embeddings**, **supervised learning**, **deep learning**, and **hyperparameter tuning**. Time-based analysis is also used for auditing and balancing, even though time is excluded from the final predictive feature set to avoid leakage.

## Conclusion

Overall, the project shows that a post’s source platform can be predicted with high accuracy from text alone, even after excluding obvious metadata shortcuts. The best results come from combining MiniLM sentence embeddings with engineered text statistics, but the simpler logistic baseline still shows that many platform differences are visible in interpretable surface features. In short, Twitter, Reddit, and Hacker News differ not just in topic, but in measurable writing style, structure, and visible platform conventions.